[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/B.%20Core%20Quay-Side%20Operations%20%28The%20Ship-to-Shore%20Interface%29/15.%20The%20Vessel%20Re-stow%20Problem%20%28Transshipment%29/P15-Tier-7_executed.ipynb)

---


# 15. The Vessel Re-stow Problem (Transshipment)
## Tier 7: Human–AI Partnership

### Purpose

This notebook demonstrates a simple human–AI decision process for a vessel re-stow scenario. It represents AI recommendations, human review, confidence- and trust-based authority allocation, and repeated decisions that update trust. The implementation is a conceptual simulation; it is not a vessel-stowage optimizer and does not establish operational performance.

### Inputs and outputs

The example creates AI agents, human experts, container records, and decision contexts. A container record includes a current position and a target position. The simulated AI counts records whose positions differ and assigns a fixed illustrative cost to each counted move. Human experts review that recommendation in the context of complexity, criticality, uncertainty, and time pressure. The decision engine combines recommendations using normalized authority weights.

The outputs are collaborative decisions, agreement levels, authority shares, trust updates, learning-outcome messages, summary metrics, and visualizations. These are outputs of the specified simulation, not measurements from a real terminal.

### Scope and modeling assumptions

- Positions are represented as tuples, but the model does not check stack access, vessel geometry, weight limits, stability, hazardous-goods rules, or whether a sequence of physical moves is feasible.
- A difference between a container's current and target position contributes one move to the abstract move count. The model does not plan intermediate moves or verify that the target positions form a feasible stowage plan.
- The illustrative cost is a fixed \$150 per counted move. It is not an estimate of actual terminal or vessel operating costs.
- Accuracy, confidence, trust, and availability are simulation parameters, not empirically calibrated probabilities. Availability is sampled randomly for each expert and decision.
- The example uses a confidence- and trust-weighted average of recommendations. A weighted average is a decision-combination rule; it does not prove that the resulting recommendation is optimal or operationally safe.
- The simulation does not train an AI model or update AI accuracy from feedback. Trust scores are updated after decisions; learning-outcome messages are descriptive prompts rather than measured learning.

### Partnership process

For each scenario, each AI agent produces an explainable recommendation. Available human experts review the first AI recommendation and may attach contextual strategy or safety checks. The engine then combines the numeric recommendations, computes agreement, updates participating experts' trust scores, and records partnership metrics. This sequence is repeated across scenarios.

Authority is allocated in proportion to confidence and trust for AI agents, and in proportion to confidence and experience for participating human experts. Let \(A\) be the set of AI agents and \(H\) the set of human experts who are available for a particular decision. For AI agent \(a\) and human expert \(h\), define the unnormalized weights

$$
u_a = c_a\,\bar{t}, \qquad
u_h = c_h\,\min\!\left(1,\frac{y_h}{20}\right),
$$

where \(c_a\) and \(c_h\) are their decision confidence values, \(\bar{t}\) is the mean trust in AI among the experts in the partnership, and \(y_h\) is the human expert's years of experience. If there are no human experts, the AI trust multiplier is set to 1. The normalized authority share for participant \(i\) is

$$
w_i = \frac{u_i}{\sum_{j\in A\cup H}u_j}.
$$

The combined move recommendation is the weighted average of participant move recommendations, rounded to an integer. The reported cost is that rounded move count multiplied by \$150. Agreement is calculated from the difference between the mean AI and human move recommendations:

$$
\text{agreement} = 1 -
\frac{|\bar{m}_{A}-\bar{m}_{H}|}
{\max(\bar{m}_{A},\bar{m}_{H},1)}.
$$

Here, \(\bar{m}_{A}\) and \(\bar{m}_{H}\) are the mean move recommendations from AI agents and available human experts. If either group is absent, the simulation assigns a neutral agreement value of 0.5. An expert's trust change after a decision is \(0.05(\text{agreement}-0.5)\), with trust kept between 0.1 and 0.95.

In the code, these calculations appear in `CollaborativeDecisionEngine._combine_decisions` and `CollaborativeDecisionEngine._calculate_trust_updates`. Across multiple scenarios, the state that carries forward is each expert's trust score and the engine's running partnership metrics. This is a sequential simulation rather than an optimization algorithm: each decision is processed once, and there is no search over alternative stowage plans.

### How to interpret the results

Agreement measures similarity in the groups' move counts; it does not measure correctness. Authority shares describe the influence assigned by the stated weighting rule, not the quality of a participant's advice. Trust changes reflect the simulation rule, not independently verified competence. Because container generation and expert availability use random draws without a fixed seed, repeated runs can produce different results.


In [ ]:
from dataclasses import dataclass
from datetime import datetime
from typing import Dict, List, Tuple
import random

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class HumanExpert:
    """A human expert participating in the decision partnership."""

    id: str
    name: str
    expertise_area: str
    experience_years: int
    trust_score: float = 0.5
    confidence_level: float = 0.7
    availability: float = 0.9


@dataclass
class AIAgent:
    """An AI agent with illustrative capability and confidence parameters."""

    id: str
    name: str
    capability: str
    accuracy: float = 0.8
    explainability_score: float = 0.7
    confidence_threshold: float = 0.8
    learning_rate: float = 0.1


@dataclass
class DecisionContext:
    """Contextual factors supplied for one simulated decision."""

    problem_id: str
    timestamp: datetime
    complexity_level: float
    criticality_level: float
    uncertainty_level: float
    time_pressure: float


@dataclass
class AIDecision:
    """An AI recommendation accompanied by an explanation."""

    agent_id: str
    recommendation: Dict
    confidence: float
    reasoning: List[str]
    supporting_data: Dict
    uncertainty_factors: List[str]


@dataclass
class HumanDecision:
    """A human review containing domain and contextual observations."""

    expert_id: str
    decision: Dict
    confidence: float
    domain_insights: List[str]
    contextual_factors: List[str]
    ethical_considerations: List[str]


@dataclass
class CollaborativeDecision:
    """The combined recommendation and its partnership information."""

    final_decision: Dict
    authority_distribution: Dict[str, float]
    agreement_level: float
    trust_update: Dict[str, float]
    learning_outcomes: List[str]
    decision_rationale: str

In [ ]:
class ExplainableAI:
    """Generate a simple move-count recommendation with explanatory text."""

    def __init__(self, agent: AIAgent):
        self.agent = agent
        self.decision_history = []
        self.explanation_templates = {
            "container_move": (
                "Moving container {container} from {source} to {target} "
                "reduces {metric} by {value}"
            ),
            "port_segregation": (
                "Segregating {port} containers to bay {bay} "
                "improves {efficiency} by {percent}%"
            ),
            "weight_balance": (
                "Rebalancing weight distribution improves vessel stability "
                "by {factor}"
            ),
            "accessibility": (
                "Repositioning containers improves accessibility for "
                "{sequence} operations"
            ),
        }

    def analyze_re_stow_problem(self, containers: List) -> AIDecision:
        """Count containers whose current and target positions differ."""
        total_moves = sum(
            container.current_position != container.target_position
            for container in containers
        )
        move_cost = total_moves * 150.0

        recommendation = {
            "total_moves": total_moves,
            "total_cost": move_cost,
            "suggested_moves": self._generate_move_suggestions(containers),
            "optimization_strategy": "cost_efficient_segregation",
        }

        reasoning = [
            f"The position comparison identifies {total_moves} containers "
            "with different current and target positions.",
            f"The illustrative estimated cost is ${move_cost:.2f} at "
            "$150 per counted move.",
            "The recommendation is a move-count estimate; it does not "
            "verify physical feasibility or port segregation.",
        ]

        uncertainty_factors = []
        if total_moves > len(containers) * 0.5:
            uncertainty_factors.append(
                "Many containers require repositioning in this abstract count."
            )
        if move_cost > 1000:
            uncertainty_factors.append(
                "The illustrative cost exceeds $1,000."
            )

        complexity = min(1.0, total_moves / max(len(containers), 1))
        confidence = max(
            0.5,
            self.agent.accuracy * (1 - complexity * 0.3),
        )

        decision = AIDecision(
            agent_id=self.agent.id,
            recommendation=recommendation,
            confidence=confidence,
            reasoning=reasoning,
            supporting_data={
                "containers_analyzed": len(containers),
                "moves_required": total_moves,
            },
            uncertainty_factors=uncertainty_factors,
        )
        self.decision_history.append(decision)
        return decision

    def _generate_move_suggestions(self, containers: List) -> List[Dict]:
        """List position differences among the first five container records."""
        suggestions = []
        for container in containers[:5]:
            if container.current_position != container.target_position:
                suggestions.append(
                    {
                        "container_id": container.id,
                        "from_position": container.current_position,
                        "to_position": container.target_position,
                        "reason": (
                            "Illustrative destination-port label: "
                            f"{container.destination_port}"
                        ),
                    }
                )
        return suggestions

    def update_from_feedback(self, feedback: Dict):
        """Apply a bounded moving update to the agent's illustrative scores."""
        if "accuracy_feedback" in feedback:
            self.agent.accuracy += self.agent.learning_rate * (
                feedback["accuracy_feedback"] - self.agent.accuracy
            )
            self.agent.accuracy = max(0.5, min(0.95, self.agent.accuracy))

        if "explainability_feedback" in feedback:
            self.agent.explainability_score += self.agent.learning_rate * (
                feedback["explainability_feedback"]
                - self.agent.explainability_score
            )
            self.agent.explainability_score = max(
                0.5, min(0.95, self.agent.explainability_score)
            )

In [ ]:
class HumanExpertSystem:
    """Represent a human review informed by expertise and context."""

    def __init__(self, expert: HumanExpert):
        self.expert = expert
        self.decision_history = []
        self.domain_knowledge = self._initialize_domain_knowledge()

    def _initialize_domain_knowledge(self) -> Dict:
        """Return illustrative knowledge statements for the expert's area."""
        knowledge = {
            "vessel_operations": {
                "weight_constraints": (
                    "Heavy containers should be placed in lower tiers "
                    "and central bays."
                ),
                "accessibility": (
                    "High-priority discharge containers should be accessible."
                ),
                "stability": (
                    "Weight distribution must maintain vessel stability."
                ),
            },
            "port_operations": {
                "segregation_efficiency": (
                    "Containers for the same port may be grouped together."
                ),
                "handling_equipment": (
                    "Crane reach and equipment limitations affect feasibility."
                ),
                "operational_flow": (
                    "Discharge-port operations should avoid interference."
                ),
            },
            "safety_compliance": {
                "weight_limits": "Position weight limits must be respected.",
                "stacking_rules": "Stacking-height restrictions must be followed.",
                "hazardous_materials": (
                    "Dangerous goods require special handling."
                ),
            },
        }
        return knowledge.get(self.expert.expertise_area, {})

    def analyze_re_stow_problem(
        self,
        containers: List,
        ai_decision: AIDecision,
        context: DecisionContext,
    ) -> HumanDecision:
        """Review the AI move count without asserting physical feasibility."""
        domain_insights = []
        contextual_factors = []
        ethical_considerations = [
            "Consider fair treatment of all shipping lines.",
            "Consider the environmental impact of re-stow operations.",
            "Maintain transparency in decision making.",
        ]

        if self.expert.expertise_area == "vessel_operations":
            domain_insights.extend(
                [
                    "Weight distribution should be reviewed for vessel stability.",
                    "Tidal conditions and vessel trim may require operational review.",
                ]
            )
        elif self.expert.expertise_area == "port_operations":
            domain_insights.extend(
                [
                    "Terminal equipment availability affects move feasibility.",
                    "Berth scheduling may constrain operations.",
                ]
            )
        elif self.expert.expertise_area == "safety_compliance":
            domain_insights.extend(
                [
                    "Moves require review against applicable safety regulations.",
                    "Container weight limits and stacking rules require verification.",
                ]
            )

        if context.time_pressure > 0.7:
            contextual_factors.append(
                "High time pressure calls for operational prioritization."
            )
        if context.criticality_level > 0.8:
            contextual_factors.append(
                "Critical operations call for safety and compliance review."
            )
        if context.uncertainty_level > 0.6:
            contextual_factors.append(
                "High uncertainty supports a cautious operational review."
            )

        human_decision = self._make_human_decision(ai_decision, context)
        experience_factor = min(1.0, self.expert.experience_years / 20)
        context_factor = 1 - (
            context.complexity_level * 0.3
            + context.uncertainty_level * 0.2
        )
        confidence = max(
            0.0,
            min(
                1.0,
                self.expert.confidence_level
                * experience_factor
                * context_factor,
            ),
        )

        decision = HumanDecision(
            expert_id=self.expert.id,
            decision=human_decision,
            confidence=confidence,
            domain_insights=domain_insights,
            contextual_factors=contextual_factors,
            ethical_considerations=ethical_considerations,
        )
        self.decision_history.append(decision)
        return decision

    def _make_human_decision(
        self,
        ai_decision: AIDecision,
        context: DecisionContext,
    ) -> Dict:
        """Add review notes while preserving the counted move requirement."""
        human_decision = ai_decision.recommendation.copy()

        if context.time_pressure > 0.7:
            human_decision["operational_priority"] = "time_priority"

        if context.criticality_level > 0.8:
            human_decision["safety_margin"] = 0.1
            human_decision["additional_checks"] = [
                "weight_verification",
                "stability_analysis",
            ]

        return human_decision

    def update_trust(self, ai_performance: float):
        """Update trust using an externally supplied performance score."""
        trust_change = 0.1 * (ai_performance - 0.7)
        self.expert.trust_score = max(
            0.1,
            min(0.95, self.expert.trust_score + trust_change),
        )

In [ ]:
class CollaborativeDecisionEngine:
    """Combine AI recommendations and available human reviews."""

    def __init__(self):
        self.ai_agents = []
        self.human_experts = []
        self.decision_history = []
        self.partnership_metrics = {
            "total_decisions": 0,
            "agreement_rate": 0.0,
            "average_trust": 0.0,
            "decision_quality": 0.0,
        }

    def add_ai_agent(self, agent: AIAgent):
        """Register an AI participant in the partnership."""
        self.ai_agents.append(agent)

    def add_human_expert(self, expert: HumanExpert):
        """Register a human participant in the partnership."""
        self.human_experts.append(expert)

    def make_collaborative_decision(
        self,
        containers: List,
        context: DecisionContext,
    ) -> CollaborativeDecision:
        """Run one recommendation, review, combination, and trust-update cycle."""
        if not self.ai_agents:
            raise ValueError("At least one AI agent is required for this process.")

        ai_decisions = [
            ExplainableAI(agent).analyze_re_stow_problem(containers)
            for agent in self.ai_agents
        ]

        human_decisions = []
        for expert in self.human_experts:
            if random.random() < expert.availability:
                human_system = HumanExpertSystem(expert)
                human_decisions.append(
                    human_system.analyze_re_stow_problem(
                        containers,
                        ai_decisions[0],
                        context,
                    )
                )

        final_decision, authority_dist, agreement = self._combine_decisions(
            ai_decisions,
            human_decisions,
            context,
        )
        trust_updates = self._calculate_trust_updates(
            ai_decisions,
            human_decisions,
            agreement,
        )
        learning_outcomes = self._generate_learning_outcomes(
            ai_decisions,
            human_decisions,
            agreement,
        )
        rationale = self._create_decision_rationale(
            ai_decisions,
            human_decisions,
            agreement,
        )

        decision = CollaborativeDecision(
            final_decision=final_decision,
            authority_distribution=authority_dist,
            agreement_level=agreement,
            trust_update=trust_updates,
            learning_outcomes=learning_outcomes,
            decision_rationale=rationale,
        )
        self.decision_history.append(decision)
        self._update_partnership_metrics(agreement, trust_updates)
        return decision

    def _combine_decisions(
        self,
        ai_decisions: List[AIDecision],
        human_decisions: List[HumanDecision],
        context: DecisionContext,
) -> Tuple[Dict, Dict, float]:
        """Normalize participant weights and combine move-count estimates."""
        authority_weights = {}
        average_trust = (
            float(np.mean([expert.trust_score for expert in self.human_experts]))
            if self.human_experts
            else 1.0
        )

        for ai_decision in ai_decisions:
            authority_weights[ai_decision.agent_id] = (
                ai_decision.confidence * average_trust
            )

        for human_decision in human_decisions:
            expert = next(
                expert
                for expert in self.human_experts
                if expert.id == human_decision.expert_id
            )
            experience_factor = min(1.0, expert.experience_years / 20)
            authority_weights[expert.id] = (
                human_decision.confidence * experience_factor
            )

        total_authority = sum(authority_weights.values())
        if total_authority <= 0:
            raise ValueError("Participant authority weights must sum to a positive value.")

        authority_weights = {
            participant_id: weight / total_authority
            for participant_id, weight in authority_weights.items()
        }

        weighted_moves = sum(
            authority_weights[decision.agent_id]
            * decision.recommendation["total_moves"]
            for decision in ai_decisions
        )
        weighted_moves += sum(
            authority_weights[decision.expert_id]
            * decision.decision.get("total_moves", 0)
            for decision in human_decisions
        )

        final_moves = int(round(weighted_moves))
        final_decision = {
            "total_moves": final_moves,
            "total_cost": final_moves * 150.0,
            "optimization_strategy": "collaborative",
        }

        if ai_decisions and human_decisions:
            ai_average = float(
                np.mean(
                    [
                        decision.recommendation["total_moves"]
                        for decision in ai_decisions
                    ]
                )
            )
            human_average = float(
                np.mean(
                    [
                        decision.decision.get("total_moves", 0)
                        for decision in human_decisions
                    ]
                )
            )
            agreement = 1.0 - abs(ai_average - human_average) / max(
                ai_average,
                human_average,
                1,
            )
        else:
            agreement = 0.5

        return final_decision, authority_weights, agreement

    def _calculate_trust_updates(
        self,
        ai_decisions: List[AIDecision],
        human_decisions: List[HumanDecision],
        agreement: float,
    ) -> Dict[str, float]:
        """Update trust for experts who participated in this decision."""
        trust_updates = {}
        for human_decision in human_decisions:
            expert = next(
                expert
                for expert in self.human_experts
                if expert.id == human_decision.expert_id
            )
            trust_change = 0.05 * (agreement - 0.5)
            trust_updates[expert.id] = trust_change
            expert.trust_score = max(
                0.1,
                min(0.95, expert.trust_score + trust_change),
            )
        return trust_updates

    def _generate_learning_outcomes(
        self,
        ai_decisions: List[AIDecision],
        human_decisions: List[HumanDecision],
        agreement: float,
    ) -> List[str]:
        """Create descriptive prompts based on confidence and agreement."""
        outcomes = []
        if agreement > 0.8:
            outcomes.append(
                "The move-count recommendations are closely aligned."
            )
        elif agreement < 0.4:
            outcomes.append(
                "The move-count recommendations differ; review assumptions and context."
            )

        if human_decisions:
            average_confidence = np.mean(
                [decision.confidence for decision in human_decisions]
            )
            if average_confidence > 0.8:
                outcomes.append(
                    "Participating human experts report high modeled confidence."
                )

        if ai_decisions:
            average_ai_confidence = np.mean(
                [decision.confidence for decision in ai_decisions]
            )
            if average_ai_confidence > 0.8:
                outcomes.append(
                    "AI agents report high modeled confidence in their recommendations."
                )
        return outcomes

    def _create_decision_rationale(
        self,
        ai_decisions: List[AIDecision],
        human_decisions: List[HumanDecision],
        agreement: float,
    ) -> str:
        """Summarize the basis of a combined recommendation."""
        rationale_parts = [
            f"Move-count agreement is {agreement:.1%}."
        ]
        if ai_decisions:
            ai_reasoning = (
                ai_decisions[0].reasoning[0]
                if ai_decisions[0].reasoning
                else "AI analysis completed."
            )
            rationale_parts.append(f"AI: {ai_reasoning}")
        if human_decisions:
            human_insight = (
                human_decisions[0].domain_insights[0]
                if human_decisions[0].domain_insights
                else "Human review completed."
            )
            rationale_parts.append(f"Human review: {human_insight}")
        return " | ".join(rationale_parts)

    def _update_partnership_metrics(
        self,
        agreement: float,
        trust_updates: Dict[str, float],
    ):
        """Update running agreement and trust averages after each decision."""
        self.partnership_metrics["total_decisions"] += 1
        number_of_decisions = self.partnership_metrics["total_decisions"]

        current_agreement = self.partnership_metrics["agreement_rate"]
        self.partnership_metrics["agreement_rate"] = (
            current_agreement * (number_of_decisions - 1) + agreement
        ) / number_of_decisions

        if self.human_experts:
            current_average_trust = self.partnership_metrics["average_trust"]
            average_trust = np.mean(
                [expert.trust_score for expert in self.human_experts]
            )
            self.partnership_metrics["average_trust"] = (
                current_average_trust * (number_of_decisions - 1)
                + average_trust
            ) / number_of_decisions

In [ ]:
def create_partnership_scenario():
    """Create the illustrative AI, expert, and container records."""
    ai_agents = [
        AIAgent(
            id="ai-001",
            name="OptimizationAI",
            capability="vessel_optimization",
            accuracy=0.85,
            explainability_score=0.75,
            confidence_threshold=0.8,
        ),
        AIAgent(
            id="ai-002",
            name="SafetyAI",
            capability="safety_compliance",
            accuracy=0.90,
            explainability_score=0.80,
            confidence_threshold=0.85,
        ),
    ]

    human_experts = [
        HumanExpert(
            id="he-001",
            name="Captain Chen",
            expertise_area="vessel_operations",
            experience_years=15,
            trust_score=0.6,
            confidence_level=0.8,
        ),
        HumanExpert(
            id="he-002",
            name="Manager Kumar",
            expertise_area="port_operations",
            experience_years=12,
            trust_score=0.7,
            confidence_level=0.75,
        ),
        HumanExpert(
            id="he-003",
            name="Inspector Smith",
            expertise_area="safety_compliance",
            experience_years=18,
            trust_score=0.65,
            confidence_level=0.85,
        ),
    ]

    containers = []
    destinations = ["HKG", "SHA", "SIN", "KOR"]
    for index in range(12):
        container = type(
            "Container",
            (),
            {
                "id": f"C{index + 1:03d}",
                "destination_port": random.choice(destinations),
                "weight": random.uniform(15, 30),
                "current_position": (
                    random.randint(1, 6),
                    random.randint(1, 4),
                    random.randint(1, 3),
                ),
                "target_position": (
                    random.randint(1, 6),
                    random.randint(1, 4),
                    random.randint(1, 3),
                ),
            },
        )()
        containers.append(container)

    return ai_agents, human_experts, containers


ai_agents, human_experts, containers = create_partnership_scenario()

print("Partnership configuration:")
print(f"  AI agents: {len(ai_agents)}")
print(f"  Human experts: {len(human_experts)}")
print(f"  Container records: {len(containers)}")

print("\nAI agents:")
for agent in ai_agents:
    print(
        f"  {agent.name}: {agent.capability} "
        f"(illustrative accuracy parameter: {agent.accuracy:.1%})"
    )

print("\nHuman experts:")
for expert in human_experts:
    print(
        f"  {expert.name}: {expert.expertise_area} "
        f"({expert.experience_years} years; "
        f"initial trust parameter: {expert.trust_score:.1%})"
    )

In [ ]:
print("\n=== INITIALIZING THE HUMAN–AI PARTNERSHIP ===")
decision_engine = CollaborativeDecisionEngine()

for agent in ai_agents:
    decision_engine.add_ai_agent(agent)

for expert in human_experts:
    decision_engine.add_human_expert(expert)

print(
    f"Partnership initialized with {len(decision_engine.ai_agents)} AI agents "
    f"and {len(decision_engine.human_experts)} human experts."
)

scenarios = [
    DecisionContext("SC-001", datetime.now(), 0.6, 0.7, 0.4, 0.5),
    DecisionContext("SC-002", datetime.now(), 0.8, 0.9, 0.6, 0.8),
    DecisionContext("SC-003", datetime.now(), 0.4, 0.5, 0.3, 0.9),
    DecisionContext("SC-004", datetime.now(), 0.7, 0.6, 0.8, 0.4),
    DecisionContext("SC-005", datetime.now(), 0.5, 0.8, 0.5, 0.6),
]

collaborative_decisions = []
print("\n=== SIMULATING COLLABORATIVE DECISIONS ===")

for scenario in scenarios:
    print(f"\nScenario: {scenario.problem_id}")
    print(
        f"  Complexity: {scenario.complexity_level:.1%}; "
        f"criticality: {scenario.criticality_level:.1%}"
    )
    print(
        f"  Uncertainty: {scenario.uncertainty_level:.1%}; "
        f"time pressure: {scenario.time_pressure:.1%}"
    )

    decision = decision_engine.make_collaborative_decision(
        containers,
        scenario,
    )
    collaborative_decisions.append(decision)

    print(
        f"  Combined recommendation: "
        f"{decision.final_decision['total_moves']} counted moves, "
        f"${decision.final_decision['total_cost']:.0f} illustrative cost"
    )
    print(f"  Move-count agreement: {decision.agreement_level:.1%}")
    print(f"  Rationale: {decision.decision_rationale}")

    if decision.trust_update:
        print("  Trust updates for participating experts:")
        for expert_id, change in decision.trust_update.items():
            expert = next(
                expert for expert in human_experts if expert.id == expert_id
            )
            print(
                f"    {expert.name}: {change:+.3f} "
                f"(trust after update: {expert.trust_score:.1%})"
            )

In [ ]:
def analyze_partnership_performance(
    decisions: List[CollaborativeDecision],
    engine: CollaborativeDecisionEngine,
    experts: List[HumanExpert],
) -> Dict:
    """Summarize the simulated decisions and the engine's running metrics."""
    if not decisions:
        raise ValueError("At least one collaborative decision is required.")

    agreements = [decision.agreement_level for decision in decisions]
    moves = [decision.final_decision["total_moves"] for decision in decisions]
    costs = [decision.final_decision["total_cost"] for decision in decisions]
    metrics = engine.partnership_metrics

    print("\n=== PARTNERSHIP PERFORMANCE SUMMARY ===")
    print("\nMove-count and cost summaries:")
    print(f"  Mean agreement: {np.mean(agreements):.1%}")
    print(f"  Agreement standard deviation: {np.std(agreements):.1%}")
    print(f"  Mean counted moves: {np.mean(moves):.1f}")
    print(f"  Mean illustrative cost: ${np.mean(costs):.0f}")
    print(f"  Cost standard deviation: ${np.std(costs):.0f}")

    print("\nCurrent trust parameters:")
    for expert in experts:
        print(f"  {expert.name}: {expert.trust_score:.1%}")

    print("\nRunning partnership metrics:")
    print(f"  Decisions processed: {metrics['total_decisions']}")
    print(f"  Mean agreement: {metrics['agreement_rate']:.1%}")
    print(f"  Mean trust: {metrics['average_trust']:.1%}")

    all_outcomes = [
        outcome
        for decision in decisions
        for outcome in decision.learning_outcomes
    ]
    unique_outcomes = list(dict.fromkeys(all_outcomes))
    print("\nDescriptive learning prompts:")
    if unique_outcomes:
        for outcome in unique_outcomes[:5]:
            print(f"  - {outcome}")
    else:
        print("  No prompts met the configured reporting conditions.")

    return {
        "avg_agreement": float(np.mean(agreements)),
        "avg_cost": float(np.mean(costs)),
        "avg_moves": float(np.mean(moves)),
        "trust_scores": [expert.trust_score for expert in experts],
        "expert_names": [expert.name for expert in experts],
        "partnership_metrics": metrics.copy(),
    }


performance_results = analyze_partnership_performance(
    collaborative_decisions,
    decision_engine,
    human_experts,
)

In [ ]:
def visualize_partnership_results(
    decisions: List[CollaborativeDecision],
    performance: Dict,
):
    """Plot the simulated agreement, trust, decisions, and partnership metrics."""
    if not decisions:
        raise ValueError("At least one collaborative decision is required.")

    figure, axes = plt.subplots(2, 3, figsize=(18, 12))

    agreements = [decision.agreement_level for decision in decisions]
    decision_numbers = range(1, len(agreements) + 1)

    axis = axes[0, 0]
    axis.plot(
        decision_numbers,
        agreements,
        marker="o",
        linewidth=2,
        label="Move-count agreement",
    )
    axis.set_title("Agreement Across Decisions", fontweight="bold")
    axis.set_xlabel("Decision number")
    axis.set_ylabel("Agreement (0–1)")
    axis.set_ylim(0, 1)
    axis.grid(True, alpha=0.3)
    axis.legend()

    axis = axes[0, 1]
    expert_names = performance["expert_names"]
    trust_scores = performance["trust_scores"]
    bars = axis.bar(expert_names, trust_scores, alpha=0.75)
    axis.set_title("Current Trust Parameters", fontweight="bold")
    axis.set_ylabel("Trust (0–1)")
    axis.set_ylim(0, 1)
    axis.tick_params(axis="x", rotation=20)
    for bar, score in zip(bars, trust_scores):
        axis.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.02,
            f"{score:.1%}",
            ha="center",
            va="bottom",
        )

    axis = axes[0, 2]
    costs = [
        decision.final_decision["total_cost"] for decision in decisions
    ]
    moves = [
        decision.final_decision["total_moves"] for decision in decisions
    ]
    scatter = axis.scatter(
        moves,
        costs,
        c=agreements,
        cmap="viridis",
        s=100,
        alpha=0.8,
    )
    axis.set_title("Illustrative Cost and Move Count", fontweight="bold")
    axis.set_xlabel("Counted moves")
    axis.set_ylabel("Illustrative cost ($)")
    colorbar = figure.colorbar(scatter, ax=axis)
    colorbar.set_label("Move-count agreement (0–1)")

    axis = axes[1, 0]
    all_authorities = [
        decision.authority_distribution for decision in decisions
    ]
    participant_ids = list(
        dict.fromkeys(
            participant_id
            for authority in all_authorities
            for participant_id in authority
        )
    )
    average_authority = [
        np.mean(
            [authority.get(participant_id, 0.0) for authority in all_authorities]
        )
        for participant_id in participant_ids
    ]
    if participant_ids:
        axis.bar(participant_ids, average_authority, alpha=0.75)
        axis.tick_params(axis="x", rotation=30)
    axis.set_title("Mean Authority Share", fontweight="bold")
    axis.set_ylabel("Mean share across decisions")

    axis = axes[1, 1]
    outcome_counts = {}
    for decision in decisions:
        for outcome in decision.learning_outcomes:
            short_outcome = outcome[:48] + "..." if len(outcome) > 48 else outcome
            outcome_counts[short_outcome] = outcome_counts.get(short_outcome, 0) + 1
    if outcome_counts:
        labels = list(outcome_counts)
        counts = list(outcome_counts.values())
        axis.barh(labels, counts, alpha=0.75)
    else:
        axis.text(0.5, 0.5, "No outcomes to display", ha="center", va="center")
    axis.set_title("Learning-Prompt Frequency", fontweight="bold")
    axis.set_xlabel("Number of decisions")

    axis = axes[1, 2]
    metrics = performance["partnership_metrics"]
    metric_names = ["Mean agreement", "Mean trust"]
    metric_values = [
        metrics["agreement_rate"],
        metrics["average_trust"],
    ]
    bars = axis.bar(metric_names, metric_values, alpha=0.75)
    axis.set_title("Running Partnership Metrics", fontweight="bold")
    axis.set_ylabel("Value (0–1)")
    axis.set_ylim(0, 1)
    for bar, value in zip(bars, metric_values):
        axis.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.02,
            f"{value:.1%}",
            ha="center",
            va="bottom",
        )

    figure.suptitle(
        "Human–AI Partnership Simulation",
        fontsize=16,
        fontweight="bold",
    )
    figure.tight_layout()
    plt.show()


visualize_partnership_results(collaborative_decisions, performance_results)

In [ ]:
def compare_with_baselines(performance: Dict):
    """Report partnership outputs and state what baseline data are unavailable."""
    print("\n=== BASELINE COMPARISON ===")
    print(
        "This notebook contains no measured AI-only, human-only, or "
        "heuristic baseline observations. Numerical comparisons would "
        "therefore be unsupported."
    )
    print("\nObserved simulation summary:")
    print(f"  Mean illustrative cost: ${performance['avg_cost']:.0f}")
    print(f"  Mean counted moves: {performance['avg_moves']:.1f}")
    print(
        "  Mean move-count agreement: "
        f"{performance['avg_agreement']:.1%}"
    )

    print("\nInterpretation limits:")
    print("  - These values summarize the configured simulation only.")
    print("  - Agreement is not a measure of decision correctness.")
    print("  - A valid baseline study requires comparable data and evaluation criteria.")
    return {"partnership_simulation": performance}


baseline_comparison = compare_with_baselines(performance_results)

## Results and interpretation

The code reports the decisions generated by the sampled container positions and expert-availability draws. Its summary statistics describe those simulated decisions; they should not be interpreted as evidence that human–AI collaboration reduces moves, improves decision quality, or increases operational safety.

### What the measures mean

- **Move-count agreement** compares the average AI and available-human move counts under the equation defined above. It does not test whether either recommendation is correct.
- **Authority distribution** shows the normalized influence assigned by the confidence-, trust-, and experience-based weights. It is not a performance ranking.
- **Trust scores** evolve according to the explicit agreement-based update rule. They are simulated state variables, not survey measurements or calibrated trust estimates.
- **Illustrative cost** is derived from the move count using the fixed \$150-per-move assumption. It is not a real cost estimate.
- **Learning prompts** are generated by threshold rules. The code does not measure whether an expert or an AI system actually learned.

### When a human–AI process may be relevant

A partnership process can be considered when operational decisions require both data-supported analysis and human review of context, safety, or organizational responsibilities. In a real deployment, roles, escalation rules, uncertainty reporting, validation procedures, and accountability would need to be specified and evaluated using appropriate operational evidence.

### Practical considerations

1. Define which recommendations an AI system may make and which decisions require human authorization.
2. Make the assumptions, input data, limitations, and uncertainty of recommendations visible to reviewers.
3. Treat trust and confidence values as quantities that require meaningful definitions and validation before operational use.
4. Keep safety, legal, and physical-feasibility checks explicit; agreement between participants does not replace them.
5. Evaluate the process against documented baselines using comparable scenarios and predefined outcome measures.
6. Record feedback and decisions if the goal is to study learning over time; this simulation alone does not implement or demonstrate mutual learning.

### Extensions within the same modeling scope

The simulation could be extended with additional contextual factors, explicit evaluation records, or more detailed operational constraints. Such extensions would require a stated data source and clear validation criteria. The present notebook intentionally focuses on the mechanics of explainable recommendations, human review, weighted authority, agreement, and trust updates rather than physical stowage optimization.